# AI of GOD 5 — DINOv2 ViT-L/14 | 5-Fold CV | High-Resolution
- **Model:** `facebook/dinov2-large` (ViT-L/14, 518×518)
- **CV:** Stratified 5-Fold
- **Metric:** Macro F1 (used for val monitoring & ensemble)
- **Weights:** saved as `fold{i}_best.pt` → zipped at the end

In [1]:
# ─── Install / imports ────────────────────────────────────────────────────────
import os, gc, zipfile, warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import timm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# ─── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    # Paths
    train_csv   = '/kaggle/input/ai-of-god-5/train.csv',     # adjust if needed
    sample_csv  = '/kaggle/input/ai-of-god-5/sample_submission.csv',
    train_dir   = '/kaggle/input/ai-of-god-5/train_images',
    test_dir    = '/kaggle/input/ai-of-god-5/test_images',
    output_dir  = '/kaggle/working',

    # Model
    model_name  = 'vit_base_patch14_dinov2.lvd142m',  # timm name for DINOv2-L
    img_size    = 308,   # DINOv2-L native resolution (multiples of 14)
    num_classes = 72,    # will be overridden after reading CSV

    # Training
    n_folds     = 5,
    epochs      = 10,
    batch_size  = 8,    # lower for 518px; bump if you have >24 GB VRAM
    grad_accum  = 4,     # effective batch = 32
    lr          = 2e-4,
    weight_decay= 1e-4,
    warmup_epochs = 1,
    seed        = 42,
    num_workers = 4,
    mixed_prec  = True,  # AMP
    label_smooth= 0.1,
)

os.makedirs(CFG['output_dir'], exist_ok=True)

# Reproducibility
def seed_everything(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

In [3]:
# ─── Auto-detect paths (handles nested Kaggle dirs) ───────────────────────────
def find_file(name, root='/kaggle/input'):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

train_csv_path  = find_file('train.csv')
sample_csv_path = find_file('sample_submission.csv')
base            = os.path.dirname(train_csv_path)
CFG['train_dir'] = os.path.join(base, 'train_images')
CFG['test_dir']  = os.path.join(base, 'test_images')

train_df = pd.read_csv(train_csv_path)
sub_df   = pd.read_csv(sample_csv_path)

classes          = sorted(train_df['animal_class'].unique())
class_to_idx     = {c: i for i, c in enumerate(classes)}
idx_to_class     = {i: c for c, i in class_to_idx.items()}
train_df['label'] = train_df['animal_class'].map(class_to_idx)
CFG['num_classes'] = len(classes)

print(f'Train: {len(train_df)} | Test: {len(sub_df)} | Classes: {CFG["num_classes"]}')

Train: 3087 | Test: 1179 | Classes: 72


In [4]:
# ─── Augmentations ────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(CFG['img_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2),
])

val_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size']),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# TTA (test-time augmentation) transforms
tta_tfms = [
    val_tfm,
    transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size']),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]

In [5]:
# ─── Dataset ──────────────────────────────────────────────────────────────────
class AnimalDataset(Dataset):
    def __init__(self, df, image_dir, transform, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(os.path.join(self.image_dir, row['image_id'])).convert('RGB')
        img   = self.transform(img)
        if self.is_test:
            return img, row['image_id']
        return img, torch.tensor(row['label'], dtype=torch.long)

In [6]:
# ─── Model ────────────────────────────────────────────────────────────────────
class DINOv2Classifier(nn.Module):
    def __init__(self, model_name, num_classes, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,      # headless
            img_size=CFG['img_size'],
        )
        feat_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)      # (B, feat_dim)
        return self.head(feats)


def build_model():
    model = DINOv2Classifier(CFG['model_name'], CFG['num_classes'])
    return model.to(device)

# Quick sanity check
_ = build_model()
print('Model OK')
del _; gc.collect(); torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Model OK


In [7]:
# ─── Training utilities ───────────────────────────────────────────────────────
def get_optimizer_scheduler(model, steps_per_epoch):
    # Differential LR: lower LR for backbone, higher for head
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.head.parameters())

    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': CFG['lr'] * 0.1},
        {'params': head_params,     'lr': CFG['lr']},
    ], weight_decay=CFG['weight_decay'])

    total_steps   = CFG['epochs'] * steps_per_epoch
    warmup_steps  = CFG['warmup_epochs'] * steps_per_epoch

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[CFG['lr'] * 0.1, CFG['lr']],
        total_steps=total_steps,
        pct_start=warmup_steps / total_steps,
        anneal_strategy='cos',
    )
    return optimizer, scheduler


criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        val_loss += criterion(logits, labels).item()
        preds = logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return val_loss / len(loader), macro_f1

In [8]:
# ─── 5-Fold Training ──────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

oof_preds  = np.zeros((len(train_df), CFG['num_classes']))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1} / {CFG["n_folds"]}')
    print(f'{"="*60}')

    fold_train = train_df.iloc[train_idx]
    fold_val   = train_df.iloc[val_idx]

    train_ds = AnimalDataset(fold_train, CFG['train_dir'], train_tfm)
    val_ds   = AnimalDataset(fold_val,   CFG['train_dir'], val_tfm)

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                              shuffle=True,  num_workers=CFG['num_workers'],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                              shuffle=False, num_workers=CFG['num_workers'],
                              pin_memory=True)

    model     = build_model()
    optimizer, scheduler = get_optimizer_scheduler(
        model,
        steps_per_epoch=len(train_loader) // CFG['grad_accum']
    )
    scaler    = torch.cuda.amp.GradScaler(enabled=CFG['mixed_prec'])

    best_f1   = 0.0
    ckpt_path = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for step, (imgs, labels) in enumerate(tqdm(train_loader,
                                                    desc=f'Epoch {epoch+1}/{CFG["epochs"]}',
                                                    leave=False)):
            imgs, labels = imgs.to(device), labels.to(device)

            with torch.cuda.amp.autocast(enabled=CFG['mixed_prec']):
                logits = model(imgs)
                loss   = criterion(logits, labels) / CFG['grad_accum']

            scaler.scale(loss).backward()
            train_loss += loss.item() * CFG['grad_accum']

            if (step + 1) % CFG['grad_accum'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        avg_train_loss = train_loss / len(train_loader)
        val_loss, val_f1 = validate(model, val_loader)

        print(f'  Epoch {epoch+1:2d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val Macro F1: {val_f1:.4f} '
              + ('  ← best ✓' if val_f1 > best_f1 else ''))

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch':         epoch + 1,
                'model_state':   model.state_dict(),
                'optimizer':     optimizer.state_dict(),
                'val_macro_f1':  best_f1,
                'classes':       classes,
                'class_to_idx':  class_to_idx,
                'fold':          fold + 1,
                'cfg':           CFG,
            }, ckpt_path)

    fold_scores.append(best_f1)
    print(f'\n  Fold {fold+1} best Macro F1: {best_f1:.4f}')

    # OOF predictions with best checkpoint
    model.load_state_dict(torch.load(ckpt_path)['model_state'])
    model.eval()
    with torch.no_grad():
        for imgs, _ in DataLoader(val_ds, batch_size=CFG['batch_size'],
                                  shuffle=False, num_workers=CFG['num_workers']):
            imgs   = imgs.to(device)
            logits = model(imgs)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            oof_preds[val_idx[:len(probs)]] = probs  # safety slice

    del model, optimizer, scheduler, scaler, train_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()

# ── Summary ──
print('\n' + '='*60)
for i, s in enumerate(fold_scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'  Mean CV Macro F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print('='*60)

oof_macro_f1 = f1_score(train_df['label'], oof_preds.argmax(1),
                         average='macro', zero_division=0)
print(f'  OOF Macro F1: {oof_macro_f1:.4f}')


  FOLD 1 / 5


Epoch 1/10:   0%|          | 0/308 [00:00<?, ?it/s]

  Epoch  1 | Train Loss: 3.8322 | Val Loss: 2.3823 | Val Macro F1: 0.5918   ← best ✓


Epoch 2/10:   0%|          | 0/308 [00:00<?, ?it/s]

  Epoch  2 | Train Loss: 1.8156 | Val Loss: 1.1883 | Val Macro F1: 0.9033   ← best ✓


...

Epoch 9/10:   0%|          | 0/308 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78996cc0ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78996cc0ee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

  Epoch  9 | Train Loss: 0.8347 | Val Loss: 0.8678 | Val Macro F1: 0.9655 


Epoch 10/10:   0%|          | 0/308 [00:00<?, ?it/s]

  Epoch 10 | Train Loss: 0.8287 | Val Loss: 0.8661 | Val Macro F1: 0.9671 

  Fold 5 best Macro F1: 0.9688

  Fold 1: 0.9787
  Fold 2: 0.9723
  Fold 3: 0.9767
  Fold 4: 0.9737
  Fold 5: 0.9688
  Mean CV Macro F1: 0.9740 ± 0.0035
  OOF Macro F1: 0.0004


In [9]:
# ─── Inference with TTA + ensemble over all 5 folds ──────────────────────────
print('\nRunning test inference...')

test_preds = np.zeros((len(sub_df), CFG['num_classes']))

for fold in range(CFG['n_folds']):
    ckpt_path = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')
    ckpt      = torch.load(ckpt_path, map_location=device)

    model = build_model()
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    fold_probs = np.zeros((len(sub_df), CFG['num_classes']))

    for tta_tfm in tta_tfms:
        test_ds     = AnimalDataset(sub_df, CFG['test_dir'], tta_tfm, is_test=True)
        test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'],
                                 shuffle=False, num_workers=CFG['num_workers'])

        tta_probs = []
        with torch.no_grad():
            for imgs, _ in tqdm(test_loader, desc=f'Fold {fold+1} TTA', leave=False):
                imgs   = imgs.to(device)
                logits = model(imgs)
                probs  = F.softmax(logits, dim=1).cpu().numpy()
                tta_probs.append(probs)
        fold_probs += np.vstack(tta_probs)

    fold_probs /= len(tta_tfms)   # average over TTA
    test_preds += fold_probs       # accumulate over folds

    del model; gc.collect(); torch.cuda.empty_cache()

test_preds /= CFG['n_folds']       # average over folds

final_labels = [idx_to_class[i] for i in test_preds.argmax(1)]
submission   = pd.DataFrame({'image_id': sub_df['image_id'], 'animal_class': final_labels})
submission.to_csv(os.path.join(CFG['output_dir'], 'submission.csv'), index=False)

print('submission.csv saved.')
print(submission.head(10))


Running test inference...


Fold 1 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 1 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 2 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 2 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

...

Fold 4 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 4 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 5 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

Fold 5 TTA:   0%|          | 0/148 [00:00<?, ?it/s]

submission.csv saved.
                               image_id animal_class
0          6eab26f8b156fda90e1bc819.png   woodpecker
1  483f71af1c174ab2a0c31a836267227d.png     antelope
2  563878772a3f47ce98cf9c10039a0219.png   chimpanzee
3  9aa782974228482390c1a237408a967a.png      hamster
4  3ed553219cf049b8bd9a65c300b02fb0.png          cat
5  90f0eb3542f9497b9423bf3a13c72626.png       donkey
6  4c4dd9ae0b624badb1f6abf6055e9b6a.png     goldfish
7  d0623952da1b456197f395755f5f4c51.png      gorilla
8  9caad778bcb64734850d8da3a9242c59.png        bison
9          19bc7b9bf852ce8bab43edc1.png   woodpecker


In [10]:
# ─── Zip all fold weights + submission ────────────────────────────────────────
zip_path = os.path.join(CFG['output_dir'], 'dinov2_weights_all_folds.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fold in range(CFG['n_folds']):
        pt_file = os.path.join(CFG['output_dir'], f'fold{fold+1}_best.pt')
        if os.path.exists(pt_file):
            zf.write(pt_file, arcname=f'fold{fold+1}_best.pt')
            print(f'  Added fold{fold+1}_best.pt  ({os.path.getsize(pt_file)/1e6:.1f} MB)')
    sub_file = os.path.join(CFG['output_dir'], 'submission.csv')
    zf.write(sub_file, arcname='submission.csv')
    print(f'  Added submission.csv')

print(f'\nZip saved: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('\n✅ All done!')

  Added fold1_best.pt  (1031.7 MB)
  Added fold2_best.pt  (1031.7 MB)
  Added fold3_best.pt  (1031.7 MB)
  Added fold4_best.pt  (1031.7 MB)
  Added fold5_best.pt  (1031.7 MB)
  Added submission.csv

Zip saved: /kaggle/working/dinov2_weights_all_folds.zip  (4738.9 MB)

✅ All done!
